1) Positiv/Negativ-Klassifikator 

Überwachte Klassifikation: Snippet + Deployment-KPIs → Label `1` (negativ/mutiert) und `0` (positiv).

Dient als Proof of Concept, ob die Snippets anhand der kpi Metriken in positiv und negatibeispiele eingeteilt werden können. 


In [ ]:
import numpy as np
import pandas as pd

import pathlib




In [ ]:
ROOT = pathlib.Path("/Users/svenniederlohner/projects/Bachelorthesis_KI_gestuetztes_deployment")
KPI_CSV_DIR = ROOT / "export_kpis" / "csv_kpis"
KPI_CSV_CANDIDATES = sorted(KPI_CSV_DIR.glob("all_projects_kpis_[0-9]*.csv"))
if not KPI_CSV_CANDIDATES:
 raise FileNotFoundError(
 "Keine Zeitstempel-KPI-CSV in export_kpis/csv_kpis/ gefunden - zuerst KPIs aus dem Bucket ziehen."
)
KPI_CSV = KPI_CSV_CANDIDATES[-1]

df = pd.read_csv(KPI_CSV)

is_neg = df["variant"].str.endswith("_neg")
neg = is_neg.sum()
pos = (~is_neg).sum()
print("Zeile insgesammt", len(df))
print("Davon positive Datensaetze :", pos)
print("Negative Datensaetze :", neg)

if neg == 0 or pos == 0:
    raise ValueError(
        "Entweder keine positiven oder negativen datensaetze vorhanden"
    ) 

2) Label & Feature Matrix 

- label is_neg -> negativ Endung -> wird zu 1 (negativ)
- base_project : Projekt ohne _neg endung  -> 0 (positiv)
- pair-id : Pärchen Gruppen -> pos+neg Code bleiben zusammen
 

In [ ]:
df["is_neg"] = df["variant"].str.endswith("_neg").astype(int)
df["project_id"] = df["variant"].str.replace("_neg","", regex=False)
df["pair_id"] = df["target_method"]

3) Training und Evaluation Datenset aufteilen 


In [ ]:
train_val = df[df["project_id"].isin(TRAINING_PROJECTS)]
test_df = df[df["project_id"].isin(EVAL_PROJECTS)]

shufflesplit = GroupShuffleSplit (n_splits=1,test_size=0.25,random_state=RANDOM_STATE)
train_idx,val_idx = next(shufflesplit.split(train_val, groups= train_val["pair_id"]))

train_df = train_val.iloc[train_idx].copy()
val_df = train_val.iloc[val_idx].copy()

print("Train-Projekte:", sorted(train_df["project_id"].unique()))
print("Val-Projekte:  ", sorted(val_df["project_id"].unique()))
print("Test (held-out):", sorted(test_df["project_id"].unique()))
print("n train / val / test:", len(train_df), len(val_df), len(test_df))

4) Preprocessing 

- Fehlende Kpi werte ersetzen über den median 
- Numerische Kpis standardisieren -> Wichtig für die Logistische Regression 
- Die Umgebung (env) wird one-hot encodiert

Der Preprocessor wird ausschließlich auf die Trainingsdaten trainiert und dann auf die Validierungs und Evaluierungsdaten angewandt. 